In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
titanic = sns.load_dataset("titanic")
titanic.head()
titanic.info()
titanic.describe()

In [ ]:
features = ["pclass","sex","fare","embarked","age"]
target = ["survived"]

In [ ]:
# missing data

from sklearn.impute import SimpleImputer

imp_median = SimpleImputer(strategy="median")
titanic[["age"]] = imp_median.fit_transform(titanic[["age"]])

imp_freq = SimpleImputer(strategy="most_frequent")
titanic[["embarked"]] = imp_freq.fit_transform(titanic[["embarked"]])

In [ ]:
titanic.head()
titanic.isnull().sum()
titanic.shape

In [ ]:
# encode 

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

titanic["sex"] = le.fit_transform(titanic[["sex"]])
titanic["embarked"] = le.fit_transform(titanic["embarked"])

In [ ]:
X = titanic[features]
Y = titanic[target]
X.head()
Y.head()


In [ ]:
x_train , x_test , y_train , y_test = train_test_split(
    X,Y,test_size=0.3, random_state=42
)

In [ ]:
x_test.head()

In [ ]:
# Decision Tree Model - no pruning

from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.fit(x_train,y_train)

In [ ]:
# Evaluate the model

from sklearn.metrics import accuracy_score

y_pred = model.predict(x_test)

print(f"Accuracy : {accuracy_score(y_test,y_pred)}")

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(18,10))
plot_tree(
    model,
    features=X.columns,
    class_names=["Died","Survive"],
    filled=True
)
plt.tight_layout()
plt.show() 

DECISION TREE with pre-pruning

In [ ]:
max_depth = [2,3,4,5,6,7,8,9,10]

for depth in max_depth:

    model = DecisionTreeClassifier(max_depth=depth)
    model.fit(x_train,y_train)

    acc = model.score(x_test,y_test)
    print(f"Accuracy : {acc} for depth : {depth}")

    if(depth==4):

        plt.figure(figsize=(18,10))
        plot_tree(
            model,
            feature_names=X.columns,
            class_names=["Died","Survived"],
            max_depth = 2
        )
        plt.tight_layout()
        plt.show()


In [ ]:
min_sample_splits = [10,15,20,25,30]

for depth in max_depth:

    model = DecisionTreeClassifier(max_depth=4,min_samples_split = min_sample_splits) # used min_samples_splits = split in video 
    model.fit(x_train,y_train)

    acc = model.score(x_test,y_test)
    print(f"Accuracy : {acc} for Sample Split : {depth}")

    if(min_sample_splits==10):

        plt.figure(figsize=(18,10))
        plot_tree(
            model,
            feature_names=X.columns,
            class_names=["Died","Survived"],
            max_depth = 2
        )
        plt.tight_layout()
        plt.show()



DECISION TREE WITH post-pruning

In [ ]:
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(x_train,y_train)


In [ ]:
path = full_tree.cost_complexity_pruning_path(x_train,y_train)
ccp_alphas = path.ccp_alphas

print(ccp_alphas)

In [ ]:
# train our model for all alpha

trees = []

for alpha in ccp_alphas:

    model = DecisionTreeClassifier(random_state=42,ccp_alpha=alpha)
    model.fit(x_train,y_train)

    acc = model.score(x_test,y_test)

    trees.append((model,acc))    

In [ ]:
best_acc = 0
best_alpha = 0

for model,alpha in trees:

    curr_acc = model.score(x_test,y_test)

    if(curr_acc>best_acc):
        best_acc=curr_acc
        best_alpha = alpha

In [ ]:
best_model = DecisionTreeClassifier(ccp_alpha=best_alpha)
best_model.fit(x_train,y_train)

In [ ]:
plt.figure(figsize=(18,10))
plot_tree(
    best_model,
    feature_names=X.columns,
    class_names=["Died","Survived"],
    filled = True
)
plt.tight_layout()
plt.show()

In [ ]:
print(best_model.score(x_test,y_test)) 